# 📉 The Earnings Call Autopsy

**Four AI analysts. One earnings call. Four wildly different conclusions.**

In this exercise you will give instructions to **four AI agents**, each analyzing the exact same earnings call transcript from a completely different perspective. Each agent will produce a BUY / HOLD / SELL recommendation.

**Your job:** Craft the prompts so your agents, *as a team*, predict what actually happened to the stock price.

**The twist:** The same data produces completely different stories depending on the framing — and that’s the whole point.

---

### How it works (30 min total)
| Step | Time | What you do |
|------|------|-------------|
| 1 | 5 min | Run the setup cell below (downloads the AI model) |
| 2 | 3 min | Read the scenario and skim the earnings call |
| 3 | 10 min | Edit the prompts for all 4 analyst agents |
| 4 | 4 min | Click *Run Analysis* and watch the agents work |
| 5 | 3 min | See the reveal: predictions vs. reality |
| 6 | 5 min | (Optional) Tweak prompts and try to beat your score |

> **You will only edit plain-English prompts — no coding required.**

---
## Step 1 — Setup
**Run this cell first, then listen to the instructor!** The model downloads in the background (~3 min on Colab T4).

In [ ]:
# =====================================================================
#  SETUP — Run this cell first, then listen to the instructor!
#  (Downloads the AI model — takes ~3 min on a free Colab T4 GPU)
# =====================================================================

!pip install transformers accelerate bitsandbytes torch -q

import torch, warnings, re
warnings.filterwarnings("ignore")

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\u2705 GPU detected: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("\u274c No GPU detected! Go to Runtime > Change runtime type > T4 GPU")
    raise SystemExit("GPU required.")

# Load model with 4-bit quantization
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print(f"\nDownloading {MODEL_ID} (4-bit quantized)...")
print("This takes ~3 minutes. Perfect time to read the scenario below!\n")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
    print(f"\u2705 Model loaded successfully!")
    print(f"   Memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")
    print(f"\n\u27a1\ufe0f  Now scroll down and read the scenario!")
except Exception as e:
    print(f"\u26a0\ufe0f  Error loading 7B model: {e}")
    print("Trying smaller fallback model...")
    MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
    print(f"\u2705 Fallback model loaded: {MODEL_ID}")

---
## Step 2 — The Scenario

### Meta Platforms — Q3 2022 Earnings Call

**Date:** October 26, 2022, after market close.

Meta (formerly Facebook) just reported its Q3 2022 results. The numbers are *mixed*: revenue actually **beat** Wall Street estimates, but something in this call spooked investors so badly that the stock had one of the largest single-day drops in market history.

**Your challenge:** Read the earnings call excerpt below. Then instruct your four AI analysts to figure out what the market will do. One of your agents might catch what the “experts” missed.

**Run the cell below** to load the transcript into memory.

In [ ]:
# =====================================================================
#  THE TRANSCRIPT — Just run this cell (no edits needed)
# =====================================================================

TRANSCRIPT = """
META PLATFORMS INC. Q3 2022 EARNINGS CALL TRANSCRIPT (EXCERPTS)
October 26, 2022

PARTICIPANTS:
Mark Zuckerberg — CEO
Susan Li — CFO
Selected Wall Street Analysts

=== CEO OPENING REMARKS ===

Mark Zuckerberg: Thank you for joining us today. I want to acknowledge upfront that
our financial results have not been where we want them to be, but I believe we are
making the right long-term investments.

Before I get to the financials, I want to share where I see things heading. I
recognize that a lot of people might disagree with this investment, but from what I
can tell, I think this is going to be a very important thing and I think it would be
a mistake for us to not focus on any of these areas, which I think are going to be
fundamentally important to the future.

We are putting ourselves in a position to be the leader in the next computing
platform — the metaverse. Reality Labs is building the future. The investments we are
making today will compound over the next decade. I understand these investments are
not popular on Wall Street right now, but I also think that people are going to look
back and see that this was the right call.

Now, on our core business: we had a solid quarter in a challenging environment. We
saw some stabilization in advertising demand, and Reels continues to grow quickly.
Our AI-driven discovery engine is now recommending about 15% of content in the
Facebook feed, and that number has more than doubled since earlier this year. Short-
form video is a headwind to revenue in the near term as we monetize Reels at a lower
rate than Feed and Stories, but we expect this to be a tailwind over time as we get
better at monetization.

On expenses, I want to be straightforward: we are going to be disciplined about
headcount. We ended the quarter with over 87,000 employees and I expect that
number to be roughly flat or slightly down over the next year. This is a period
of significant efficiency focus for us.

=== CFO FINANCIAL OVERVIEW ===

Susan Li: Total revenue was $27.71 billion, down 4% year over year but above
consensus estimates of $27.4 billion. Family of Apps revenue was $27.24 billion.
Reality Labs revenue was $285 million.

Total expenses were $22.1 billion, up 19% year over year. This includes $3.67 billion
in Reality Labs operating losses for the quarter. For the full year, we expect Reality
Labs losses to grow significantly in 2023.

Net income was $4.4 billion, down 52% year over year. Earnings per share were $1.64,
which was above street estimates of $1.58.

On capital expenditures: we spent $9.4 billion in Q3 and now expect 2022 total CapEx
to be $32 to $33 billion. For 2023, we are projecting CapEx of $34 to $39 billion.
This increase is driven by investments in both our AI infrastructure to support our
discovery engine, Reels, and ads systems, as well as Reality Labs hardware and
software development.

For Q4, we expect revenue to be in the range of $30 to $32.5 billion. Total expenses
for 2023 are expected to be $96 to $101 billion, compared to our prior estimate of
$85 to $87 billion.

=== ANALYST Q&A (SELECTED) ===

Brian Nowak (Morgan Stanley): Mark, on the Reality Labs side, you mentioned losses
growing in 2023. Can you help us understand when you expect this segment to reach
profitability, or at least when losses will plateau?

Mark Zuckerberg: I appreciate the question. I'm not going to put a specific timeline
on profitability for Reality Labs. This is a massive opportunity and we are investing
accordingly. I know this isn't what Wall Street wants to hear, but I think this is
one of the most important things we can be working on.

Susan Li: I'll add that the 2023 expense increase reflects both continued Reality
Labs investment and also significant infrastructure spend for AI. We are not breaking
out the split between AI infrastructure and metaverse investment at this time.

Eric Sheridan (Goldman Sachs): Susan, can you walk us through the 2023 expense
guidance? The $96 to $101 billion range is a pretty significant jump from the $85
to $87 billion you guided last quarter.

Susan Li: Correct. The increase is driven by several factors: higher infrastructure
costs for our AI workloads, continued investment in Reality Labs, and some increases
in data center and facilities costs. We are being transparent about these costs
because we think it's important for investors to understand the magnitude of the
bets we are making.

Mark Nowak (Evercore): Mark, users are spending more time on Reels but you mentioned
it monetizes at a lower rate. When will that gap close?

Mark Zuckerberg: Reels is on a trajectory similar to Stories, which also started
monetizing below Feed rates. We think the gap closes over the next 12 to 18 months
for the most part. The engagement trends are very strong. But near-term, yes, it is
a headwind to revenue as people shift time from Feed to Reels. We view this as a
necessary transition.

Justin Post (Bank of America): Mark, with the macro environment where it is, some
investors might argue this is the wrong time to be increasing spending so
aggressively. How do you respond to that?

Mark Zuckerberg: Look, I get that this is a difficult environment. But I also think
that this is actually the time to double down. History shows that the companies that
invest through downturns come out stronger. We have the balance sheet to do this.
I'm not going to pretend this is easy, but I am confident in the direction.

I want to be clear about something: we are not just spending money to spend money.
Every dollar going into Reality Labs is carefully allocated. We have a clear
technological roadmap. But I'm also realistic — not every bet will work. Some of
these investments will fail. That's the nature of doing cutting-edge research.

Dave Wehner (outgoing CFO, closing comments): I'll just add for context that our
Family of Apps remains one of the most profitable businesses in the world, generating
over $10 billion in operating income this quarter alone. We have significant financial
flexibility.
"""

# What ACTUALLY happened to the stock
ACTUAL_OUTCOME = {
    "company": "Meta Platforms",
    "quarter": "Q3 2022",
    "stock_move_1day": -24.6,
    "stock_move_5day": -20.1,
    "direction": "DOWN",
    "key_driver": (
        "Investors spooked by $15B+ annual metaverse spending, 2023 expense guidance "
        "jumped $11-14B above prior estimates, and Zuckerberg refused to put a "
        "profitability timeline on Reality Labs — despite revenue beating expectations."
    ),
}

print(f"\u2705 Transcript loaded: {ACTUAL_OUTCOME['company']} {ACTUAL_OUTCOME['quarter']} earnings call")
print(f"   Transcript length: {len(TRANSCRIPT.split())} words")
print(f"\n\u27a1\ufe0f  Read the transcript above, then scroll down to edit the agent prompts.")

---
## Infrastructure (just run — no edits needed)

In [ ]:
# @title Helper Functions (run this cell — no edits needed)

import re, time, textwrap
from IPython.display import display, HTML, Markdown

# ---------- Output format instructions appended to every agent prompt ----------
OUTPUT_FORMAT = """

After your analysis, you MUST end your response with EXACTLY this format:

RECOMMENDATION: BUY/HOLD/SELL
CONFIDENCE: [number 0-100]
PREDICTED_MOVE: [signed percentage, e.g. +5.2 or -12.0]
KEY_FINDINGS:
- [finding 1]
- [finding 2]
- [finding 3]
ONE_LINER: [your one-sentence summary]
"""

# ---------- Parse model output with regex ----------
def _parse_output(text):
    """Parse structured fields from model output. Returns dict or None."""
    rec_m = re.search(r"RECOMMENDATION:\s*(BUY|HOLD|SELL)", text, re.IGNORECASE)
    conf_m = re.search(r"CONFIDENCE:\s*(\d+)", text)
    move_m = re.search(r"PREDICTED_MOVE:\s*([+-]?\d+\.?\d*)", text)
    liner_m = re.search(r"ONE_LINER:\s*(.+)", text)

    # Key findings: capture lines starting with - after KEY_FINDINGS:
    findings = []
    kf_m = re.search(r"KEY_FINDINGS:\s*\n((?:\s*-.*\n?)+)", text)
    if kf_m:
        findings = [line.strip().lstrip("- ").strip() for line in kf_m.group(1).strip().split("\n") if line.strip().startswith("-")]

    if rec_m and conf_m and move_m:
        return {
            "recommendation": rec_m.group(1).upper(),
            "confidence": min(100, max(0, int(conf_m.group(1)))),
            "predicted_move": float(move_m.group(1)),
            "key_findings": findings if findings else ["(see full analysis above)"],
            "one_liner": liner_m.group(1).strip() if liner_m else "(no summary provided)",
        }
    return None

# ---------- Run one analyst agent ----------
def run_analyst(system_prompt, transcript, agent_name="Agent"):
    """Send the transcript to the model with the analyst's system prompt."""
    full_system = system_prompt.strip() + OUTPUT_FORMAT

    messages = [
        {"role": "system", "content": full_system},
        {"role": "user", "content": f"Analyze the following earnings call transcript:\n\n{transcript}"},
    ]

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )

    # Decode only the generated tokens
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(generated, skip_special_tokens=True)

    result = _parse_output(raw_text)

    # Retry once if parsing failed
    if result is None:
        retry_messages = messages + [
            {"role": "assistant", "content": raw_text},
            {"role": "user", "content": (
                "Your response could not be parsed. Please respond ONLY with the "
                "structured format:\n"
                "RECOMMENDATION: BUY/HOLD/SELL\n"
                "CONFIDENCE: [0-100]\n"
                "PREDICTED_MOVE: [e.g. +5.0 or -12.0]\n"
                "KEY_FINDINGS:\n- finding 1\n- finding 2\n- finding 3\n"
                "ONE_LINER: [summary]"
            )},
        ]
        retry_text = tokenizer.apply_chat_template(retry_messages, tokenize=False, add_generation_prompt=True)
        retry_inputs = tokenizer(retry_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            retry_ids = model.generate(**retry_inputs, max_new_tokens=300, temperature=0.3, do_sample=True)
        retry_gen = retry_ids[0][retry_inputs["input_ids"].shape[1]:]
        raw_text_2 = tokenizer.decode(retry_gen, skip_special_tokens=True)
        result = _parse_output(raw_text_2)
        if result is not None:
            result["raw_text"] = raw_text_2
            return result

    if result is None:
        # Fallback defaults
        result = {
            "recommendation": "HOLD",
            "confidence": 50,
            "predicted_move": 0.0,
            "key_findings": ["(parsing issue \u2014 rerun for better results)"],
            "one_liner": "(could not parse agent output)",
            "parse_failed": True,
        }

    result["raw_text"] = raw_text
    return result

# ---------- Scoring ----------
def score_analyst(prediction, actual_move):
    """Score a single agent's prediction. Returns 0-100."""
    actual_direction = "UP" if actual_move > 2 else ("DOWN" if actual_move < -2 else "FLAT")
    pred_direction = (
        "UP" if prediction["predicted_move"] > 2
        else ("DOWN" if prediction["predicted_move"] < -2 else "FLAT")
    )

    score = 0

    # Direction accuracy (40 points)
    if pred_direction == actual_direction:
        score += 40
    elif pred_direction == "FLAT":
        score += 20  # partial credit for hedging

    # Magnitude accuracy (40 points)
    distance = abs(prediction["predicted_move"] - actual_move)
    magnitude_score = max(0, 40 - distance * 2)  # lose 2 pts per % off
    score += magnitude_score

    # Confidence calibration (20 points)
    if pred_direction == actual_direction:
        score += prediction["confidence"] / 100 * 20
    else:
        score += (100 - prediction["confidence"]) / 100 * 20

    return round(score)

def total_score(all_results, actual_move):
    """Average across all agents."""
    return round(sum(score_analyst(r, actual_move) for r in all_results) / len(all_results))

def get_rating(score):
    if score >= 85: return "Hedge Fund Material"
    if score >= 70: return "Promising Prompt Engineer"
    if score >= 55: return "The Signals Were There"
    if score >= 40: return "Back to the Drawing Board"
    return "Your Agents Need Therapy"

# ---------- Rich display ----------
AGENT_COLORS = {
    "Bull": "#2e7d32",
    "Short Seller": "#c62828",
    "Journalist": "#e65100",
    "Regulator": "#1565c0",
}

def display_report(agent_name, result):
    color = AGENT_COLORS.get(agent_name, "#333")
    findings_html = "".join(f"<li>{f}</li>" for f in result["key_findings"])
    parse_warning = '<p style="color:orange;">\u26a0\ufe0f Parsing issue \u2014 rerun for better results</p>' if result.get("parse_failed") else ""
    html = f"""
    <div style="border:2px solid {color}; border-radius:10px; padding:16px; margin:10px 0;">
        <h3 style="color:{color}; margin-top:0;">{agent_name.upper()} ANALYST REPORT</h3>
        {parse_warning}
        <table style="font-size:15px;">
            <tr><td><b>Recommendation:</b></td><td style="font-size:18px; font-weight:bold;">{result['recommendation']}</td></tr>
            <tr><td><b>Confidence:</b></td><td>{result['confidence']}/100</td></tr>
            <tr><td><b>Predicted Move:</b></td><td>{result['predicted_move']:+.1f}%</td></tr>
        </table>
        <p><b>Key Findings:</b></p>
        <ul>{findings_html}</ul>
        <p><b>One-Liner:</b> <em>\"{result['one_liner']}\"</em></p>
    </div>
    """
    display(HTML(html))

def display_scoreboard(results_dict, actual):
    actual_move = actual["stock_move_1day"]
    rows = ""
    best_agent = None
    best_score = -1
    for name, res in results_dict.items():
        s = score_analyst(res, actual_move)
        distance = abs(res["predicted_move"] - actual_move)
        label = "CLOSE" if distance <= 10 else "MISS"
        if s > best_score:
            best_score = s
            best_agent = name
        rows += f"""
        <tr>
            <td>{name}</td>
            <td style="font-weight:bold;">{res['recommendation']}</td>
            <td>{res['confidence']}%</td>
            <td>{res['predicted_move']:+.1f}%</td>
            <td>{label} ({distance:.1f} pts off)</td>
            <td style="font-weight:bold;">{s}/100</td>
        </tr>"""

    ts = total_score(list(results_dict.values()), actual_move)
    rating = get_rating(ts)

    # Check for easter eggs
    recs = [r["recommendation"] for r in results_dict.values()]
    easter_egg = ""
    if all(r == "BUY" for r in recs):
        easter_egg = '<p style="color:#e65100;">\U0001f914 Unanimous bullishness is usually a contrarian sell signal...</p>'
    elif all(r == "SELL" for r in recs):
        easter_egg = '<p style="color:#e65100;">\U0001f914 When everyone agrees, everyone is usually wrong... or it\'s Enron.</p>'

    html = f"""
    <div style="border:3px solid #333; border-radius:10px; padding:20px; margin:20px 0; background:#fafafa;">
        <h2 style="margin-top:0;">PREDICTIONS vs REALITY</h2>
        <p style="font-size:18px;">Actual stock move: <b style="color:#c62828;">{actual_move:+.1f}%</b> (next day)</p>
        <table style="width:100%; border-collapse:collapse; font-size:14px;">
            <tr style="border-bottom:2px solid #333; font-weight:bold;">
                <td>Agent</td><td>Rec</td><td>Confidence</td><td>Predicted</td><td>Accuracy</td><td>Score</td>
            </tr>
            {rows}
        </table>
        {easter_egg}
        <hr>
        <p style="font-size:13px;">\U0001f3c6 <b>Best Agent: {best_agent}</b> (score: {best_score}/100)</p>
    </div>

    <div style="border:3px solid #1565c0; border-radius:10px; padding:20px; margin:20px 0; background:#e3f2fd; font-family:monospace;">
        <pre style="margin:0; font-size:14px; color:#0d47a1;">
\u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510
\u2502  BLOOMBERG TERMINAL \u2014 ANALYST SCORECARD   \u2502
\u251c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2524
\u2502  YOUR TOTAL ACCURACY SCORE:  {ts:>3}/100       \u2502
\u2502  RATING: {rating:<37s} \u2502
\u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518
        </pre>
    </div>
    """
    display(HTML(html))

print("\u2705 Helper functions loaded.")

---
## Step 3 — Edit the Agent Prompts

Below are **four AI analyst agents**. Each one will read the same earnings call transcript but from a different perspective.

**Your job:** Edit the prompt in each cell to make the agent as insightful as possible. The better your prompts, the better your score!

> Think about: What should each agent focus on? What signals matter? How should they weigh different information?

In [ ]:
# ================================================================
# 🎯 AGENT 1: THE BULL
#  Edit the prompt below to make this agent find bullish signals.
#  The better your prompt, the more accurate the analysis!
# ================================================================

bull_prompt = """
You are a bullish equity research analyst. You believe in this company's
long-term story. Analyze the earnings call transcript and find reasons
to be optimistic.

[Improve this prompt! Think about: What specific signals should the bull
look for? How should it weigh revenue vs. guidance vs. management tone?
Should it compare to market expectations?]
"""

In [ ]:
# ================================================================
# 🎯 AGENT 2: THE SHORT SELLER
#  Edit the prompt below to make this agent find bearish signals.
# ================================================================

bear_prompt = """
You are a skeptical short seller looking for reasons this stock will drop.
Analyze the earnings call for red flags, deceptive language, and hidden risks.

[Improve this prompt! Think about: What verbal cues indicate management
is hiding something? What financial patterns are concerning? What questions
did management dodge?]
"""

In [ ]:
# ================================================================
# 🎯 AGENT 3: THE JOURNALIST
#  Edit the prompt below. Journalists often predict market moves
#  better than analysts because they think about NARRATIVE, not numbers.
# ================================================================

journalist_prompt = """
You are a financial journalist writing a breaking story about this
earnings call. Find the most newsworthy angle — the headline that will
get clicks and move the market.

[Improve this prompt! Think about: What makes a story go viral?
What angle would CNBC lead with? What would scare retail investors?]
"""

In [ ]:
# ================================================================
# 🎯 AGENT 4: THE REGULATOR
#  Edit the prompt below. Regulators look for what's NOT said
#  as much as what IS said.
# ================================================================

regulator_prompt = """
You are an SEC examiner reviewing this earnings call for potentially
misleading statements, omissions, or forward-looking statements that
lack adequate qualification.

[Improve this prompt! Think about: What legal/regulatory red flags
exist? Is management being evasive? Are projections qualified properly?]
"""

---
## Step 4 — Run the Analysis

All four agents will now analyze the same earnings call transcript. This takes ~3 minutes.

In [ ]:
# =====================================================================
#  RUN ANALYSIS — Click the play button!
# =====================================================================

from IPython.display import clear_output
import time

agents = [
    ("Bull", bull_prompt),
    ("Short Seller", bear_prompt),
    ("Journalist", journalist_prompt),
    ("Regulator", regulator_prompt),
]

results = {}
total_start = time.time()

print("\U0001f680 Starting analysis...\n")

for i, (name, prompt) in enumerate(agents, 1):
    print(f"[{i}/4] Running {name} analysis...", end=" ", flush=True)
    start = time.time()
    result = run_analyst(prompt, TRANSCRIPT, agent_name=name)
    elapsed = time.time() - start
    results[name] = result
    print(f"Done! ({elapsed:.0f}s) \u2192 {result['recommendation']}")

total_elapsed = time.time() - total_start
print(f"\n\u2705 All 4 agents complete in {total_elapsed:.0f} seconds.\n")
print("=" * 60)

# Display each report
for name, result in results.items():
    display_report(name, result)

# Journalist headline award
if "Journalist" in results:
    display(HTML(f"""
    <div style="border:2px dashed #e65100; border-radius:10px; padding:12px; margin:10px 0; background:#fff3e0; text-align:center;">
        <p style="font-size:16px; margin:0;">\U0001f3c6 <b>BEST HEADLINE AWARD</b> \U0001f3c6</p>
        <p style="font-size:20px; font-style:italic; margin:8px 0;">\"{results['Journalist']['one_liner']}\"</p>
        <p style="font-size:12px; color:#666; margin:0;">\u2014 The Journalist Agent</p>
    </div>
    """))

print("\n\u27a1\ufe0f  Run the next cell to see how your agents did vs. reality!")

---
## Step 5 — The Reveal

In [ ]:
# =====================================================================
#  THE REVEAL — How did your agents do?
# =====================================================================

from IPython.display import display, HTML, Markdown

# Show the scoreboard
display_scoreboard(results, ACTUAL_OUTCOME)

# Score breakdown
actual_move = ACTUAL_OUTCOME["stock_move_1day"]
actual_dir = "DOWN"

direction_correct = sum(
    1 for r in results.values()
    if ("DOWN" if r["predicted_move"] < -2 else ("UP" if r["predicted_move"] > 2 else "FLAT")) == actual_dir
)

avg_distance = sum(abs(r["predicted_move"] - actual_move) for r in results.values()) / len(results)

ts = total_score(list(results.values()), actual_move)

print(f"""\nSCORE BREAKDOWN\n{"=" * 40}""")
print(f"Direction accuracy:  {direction_correct}/4 agents got direction right")
print(f"Avg distance:        {avg_distance:.1f} percentage points off")
print(f"Overall score:       {ts}/100")
print(f"Rating:              \"{get_rating(ts)}\"")

# The actual outcome reveal
display(HTML(f"""
<div style="border:3px solid #c62828; border-radius:10px; padding:20px; margin:20px 0; background:#ffebee;">
    <h2 style="color:#c62828; margin-top:0;">WHAT ACTUALLY HAPPENED</h2>
    <p style="font-size:18px;"><b>{ACTUAL_OUTCOME['company']}</b> stock dropped
    <b style="color:#c62828; font-size:24px;">{ACTUAL_OUTCOME['stock_move_1day']}%</b>
    the next day \u2014 one of the largest single-day drops in market history.</p>
    <p style="font-size:15px;"><b>5-day move:</b> {ACTUAL_OUTCOME['stock_move_5day']}%</p>
    <p style="font-size:15px;"><b>Why?</b> {ACTUAL_OUTCOME['key_driver']}</p>
    <p style="font-size:14px; color:#666;">The stock went from ~$130 to ~$98 overnight. Over $80 billion
    in market cap evaporated in a single trading session.</p>
</div>
"""))

# The twist / lesson
display(Markdown("""
---

### THE SURPRISING TRUTH

In exercises like this, the **Journalist agent** often turns out to be the most accurate predictor. Why?

Because markets don't move on fundamentals alone — they move on **narratives**. The journalist's job is to find the story that will *spread*, and that's exactly what moves stock prices in the short term.

In real studies, Bloomberg terminal headlines predict next-day stock moves better than analyst models. **The story IS the signal.**

### LESSON FOR YOUR BUSINESS

When you deploy AI agents, **the framing and perspective you give them matters as much as the data.** The same information, analyzed from different angles, produces completely different conclusions.

This is both the power and the danger of agentic AI:
- **Power:** You can explore every angle of a decision simultaneously
- **Danger:** The prompt shapes the conclusion — and whoever writes the prompt controls the narrative

The four agents read the *exact same transcript*. The only thing that differed was the **perspective you gave them**. That's prompt engineering in a nutshell.
"""))

print("\n" + "=" * 60)
print("Want to try again? Edit the prompts above and re-run the")
print("analysis cell. The model stays loaded \u2014 no need to re-download!")
print("=" * 60)